In [17]:
import pickle
import numpy as np
import pandas as pd
import os
from iblutil.numerical import bincount2D
from iblatlas.atlas import BrainRegions
from datetime import datetime

from one.api import ONE
one= ONE()

In [2]:
prefix = '/home/ines/repositories/'
# prefix = '/Users/ineslaranjeira/Documents/Repositories/'

# Load data

In [3]:
import brainwidemap
# this dataframe holds a summary of all the sessions
# and for us importantly, the eids and pids
bwm_df = brainwidemap.bwm_query()  # each row of this dataframe is a recording

n_sessions = bwm_df["eid"].unique().shape[0]
n_insertions = bwm_df["pid"].unique().shape[0]
print(
    f"{n_sessions} sessions with {n_insertions} individual neuropixel recordings"
)
# bwm_df.head()
bwm_pid = bwm_df['pid'].unique()

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
459 sessions with 699 individual neuropixel recordings


## LDA axis

In [4]:
# cluster_df = pd.read_pickle(data_path+'extended_mouse_LDA_5_bins_cut')
data_path = prefix + 'representation_learning_variability/paper-individuality/clustering/'
# session_cluster = pd.read_parquet(data_path+'cluster_per_session')
lda = pd.read_pickle(data_path+'extended_mouse_LDA_5_bins_cut')
lda = pd.read_pickle(data_path+'mouse_LDA_5_bins_cut18-06-2026')
lda = lda.rename(columns={0: 'lda_1'})

lda_eid = lda.loc[lda['session'].isin(list(bwm_df.eid)), 'session']
lda_pid = bwm_df.loc[bwm_df['eid'].isin(lda_eid), 'pid']

In [5]:
print(len(lda_eid))
print(len(lda_pid))

244
380


## Available neural files

In [7]:
save_states_path = prefix + 'representation_learning_variability/paper-individuality/data/neuron_files/'
# save_states_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/neuron_files/'

filepaths = [os.path.join(save_states_path, filename) for filename in os.listdir(save_states_path)]
print(len(filepaths))

464


# Fano factor

In [ ]:
# Initialize IBL Brain Atlas and target regions
BRAIN_REGIONS = ['VISa', 'VISam', 'CA1', 'DG', 'LP', 'PO']
regions = BrainRegions()

def get_simplified_area(col_name, filter_sessions=False):
    """Traces an Allen ID/acronym back up to our 6 target regions."""
    raw_acronym = col_name.split('_neuron_')[0]
    allen_ids = regions.acronym2id(raw_acronym)
    beryl_ids = regions.remap(allen_ids, source_map='Allen', target_map='Beryl')
    ancestor = regions.id2acronym(beryl_ids)[0]
    if filter_sessions:
        if ancestor in BRAIN_REGIONS:
            return ancestor
        else:
            return None
    else:
        return ancestor

def get_psth(df, col_name, events, pre=.5, post=1):
    all_trials = []
    # Explicitly tracking valid event indices to match with condition labels downstream
    valid_event_indices = []
    
    for idx, event in enumerate(events):
        # Define window
        start = event - pre
        end = event + post
        # Extract data for the specified neuron column
        window = df.loc[(df['Bin'] >= start) & (df['Bin'] < end), col_name]
        if len(window) == int((pre + post) * 60):  # Assuming 60 Hz sampling rate
            all_trials.append(window.values)
            valid_event_indices.append(idx)
            
    return np.array(all_trials), valid_event_indices

In [14]:
relevant_pids = lda_pid.copy()

In [15]:
# Sliding window parameters based on 60 Hz sampling rate
WINDOW_WIDTH_BINS = int(0.1 * 60)   # 100 ms
WINDOW_STEP_BINS = max(1, int(0.02 * 60)) # 20 ms
MAX_TRIALS_TO_PROCESS = 200 
# path_dir = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/neuron_files/'
path_dir = prefix + 'representation_learning_variability/paper-individuality/data/neuron_files/'

summary_records = []
for pid in relevant_pids:
    filepath = os.path.join(path_dir, f'states_neurons_file_{pid}')
    try:
        with open(filepath, 'rb') as f:
            raw_data = pickle.load(f)
            
        state_with_spikes = raw_data.dropna(subset=['Bin', 'most_likely_states']).reset_index(drop=True)
        if state_with_spikes.empty:
            continue
            
        pid_name = state_with_spikes['session_pid'].iloc[0]
        session_name = state_with_spikes['session'].iloc[0]
        
        # --- INFER STIMULUS CONDITION PER TRIAL ---
        def infer_side(row):
            if row['correct'] == 1:
                return "Right" if row['choice'] == 'left' else "Left"
            else:
                return "Left" if row['choice'] == 'right' else "Right"

        state_with_spikes['stim_side'] = state_with_spikes.apply(infer_side, axis=1)
        state_with_spikes['condition'] = state_with_spikes['stim_side'] + "_" + state_with_spikes['contrast'].astype(str)
        
        # Pull chronologically sorted trials, capped at MAX_TRIALS_TO_PROCESS
        all_events = np.array(state_with_spikes['goCueTrigger_times'].unique())
        events_subset = all_events[:MAX_TRIALS_TO_PROCESS]
        
        # Extract condition strings specifically for our trial subset
        trial_metadata = state_with_spikes.drop_duplicates(subset=['goCueTrigger_times'])
        trial_conditions_subset = trial_metadata.loc[
            trial_metadata['goCueTrigger_times'].isin(events_subset), 'condition'
        ].values
        
        spike_columns = [col for col in state_with_spikes.columns if '_spike_count' in col]
        
        # Process each neuron completely independently
        for col in spike_columns:
            area = get_simplified_area(col, filter_sessions=False)
            if not area:  # Skip neurons that don't belong to your target BRAIN_REGIONS
                continue
                
            # Extract PSTH for this individual neuron: Shape (n_trials, n_time_bins)
            neuron_psth, valid_idx = get_psth(state_with_spikes, col, events_subset, pre=.5, post=1)
            if len(neuron_psth) < 10: # Skip neuron if it has too few trials
                continue
                
            n_trials, n_time_bins = neuron_psth.shape
            
            # Filter condition labels to exactly match this neuron's valid trials
            neuron_conditions = trial_conditions_subset[valid_idx]
            unique_conditions = np.unique(neuron_conditions)
            
            # Setup sliding windows
            window_starts = list(range(0, n_time_bins - WINDOW_WIDTH_BINS + 1, WINDOW_STEP_BINS))
            
            # Calculate condition-adjusted Fano Factor profile for this single neuron
            ff_profile = []
            for start_b in window_starts:
                end_b = start_b + WINDOW_WIDTH_BINS
                window_counts = neuron_psth[:, start_b:end_b].sum(axis=1)
                
                residuals_union = []
                weighted_mean_denom = 0
                
                for cond in unique_conditions:
                    cond_idx = np.where(neuron_conditions == cond)[0]
                    if len(cond_idx) == 0:
                        continue
                        
                    cond_counts = window_counts[cond_idx]
                    cond_mean = np.mean(cond_counts)
                    
                    residuals_union.extend(cond_counts - cond_mean)
                    weighted_mean_denom += len(cond_idx) * cond_mean
                    
                final_weighted_mean = weighted_mean_denom / n_trials
                residual_variance = np.var(residuals_union)
                
                ff_val = residual_variance / (final_weighted_mean + 1e-6)
                ff_profile.append(ff_val)
            
            # Build record per NEURON instead of per brain region
            record = {
                'pid': pid_name,
                'session': session_name,
                'neuron_id': col, # Retain the specific neuron name/cluster ID
                'area': area,
                'n_trials': n_trials
            }
            for w_idx, ff_val in enumerate(ff_profile):
                record[f'time_{w_idx}'] = ff_val
                
            summary_records.append(record)
            
        print(f"Processed PID: {pid_name}")
        
    except Exception as e:
        print(f"Error processing {filepath}: {e}")

# Compile final summary dataframe where each row is a single cell
summary_df = pd.DataFrame(summary_records)

Processed PID: 7b05cccc-44f6-4491-a0ea-e38d6e95513d
Processed PID: 99993a2b-588e-4c0c-bfec-e3dfb4f61534
Processed PID: 298e2a70-9801-45f0-b91c-d6bb9718427e
Processed PID: 9dff5593-e781-41a3-a6f9-20d06085e4f8
Processed PID: 0909252c-3ad0-413f-96f5-7eff885b50aa
Processed PID: e45a00b1-14a0-4f5e-9ea5-9f76d042b11c
Processed PID: 117f0d28-3cc0-4837-9e3e-46db5bc3e662
Processed PID: 85bdeae3-269b-4e39-bd9b-2b0d95aff2fa
Processed PID: 01864e9d-0dbe-41d4-9e3a-0285348ecfc1
Processed PID: a9e83d8a-7c90-4152-abad-53a1ad94d73a
Processed PID: 0777b1bf-964b-49b7-888b-8a6c9df09c3b
Processed PID: 9793c99d-f918-4931-8bba-fdb978bd8e0a
Processed PID: 08ed0b3c-9f94-4c1f-8522-3d42a642a6b0
Processed PID: a85b9795-f99c-4c1d-a376-8b5ef095ffd7
Processed PID: 43b9b189-5221-46a0-928a-e137bc326534
Processed PID: c410fda5-a6db-4697-a7e3-6ed65601844d
Processed PID: ce0dc660-f19e-46a3-94f9-646bebae6805
Processed PID: ef7e9f3e-a53a-4e00-8f6a-537667af2bea
Processed PID: 4431f9fd-aaa8-4b10-905c-0c6869ea1088
Processed PI

In [18]:
save_path = prefix + 'representation_learning_variability/paper-individuality/4_mice/'
current_date = datetime.now().strftime('%d-%m-%Y')
filename = f'psths_fanofactor_per_neuron_all_regions_{current_date}'
summary_df.to_parquet(save_path + filename, compression='gzip')

In [200]:
save_path = prefix + 'representation_learning_variability/paper-individuality/4_mice/'
summary_df.to_pickle(save_path+'psths_fanofactor_per_neuron_all_regions')